In [7]:
import pygame, random, sys

# Written by student, structure and game loop logic assisted by ChatGPT

pygame.init()
W, H = 800, 300
screen = pygame.display.set_mode((W, H))
pygame.display.set_caption("Endless Runner")
clock = pygame.time.Clock()

# --- Constants ---
GROUND = 240
GRAVITY = 0.8
JUMP_VY = -15
BASE_SPEED = 6

# --- Colours ---
SKY       = (135, 206, 235)
GRASS     = (80,  160,  80)
PLAYER_C  = (70,  130, 200)
OBS_C     = (60,  150,  60)   # ground obstacle
OBS2_C    = (0,    0,    0)   # floating obstacle 
HEART_C   = (220,  50,  50)
WHITE     = (255, 255, 255)
BLACK     = (0,   0,   0)
RED       = (200,  0,   0)
DARK      = (30,   30,  30)

font_big  = pygame.font.SysFont(None, 64)
font_med  = pygame.font.SysFont(None, 36)
font_sm   = pygame.font.SysFont(None, 26)

# --- Game state ---
# States: "menu", "playing", "dead"
state       = "menu"
high_score  = 0   # session only

def make_player():
    return pygame.Rect(100, GROUND - 50, 40, 50)

def reset_game():
    p      = make_player()
    return {
        "player":    p,
        "vy":        0,
        "on_ground": True,
        "obstacles": [],
        "score":     0,
        "speed":     BASE_SPEED,
        "spawn":     0,
        "lives":     3,
        "invincible":0,   # frames of invincibility remaining
    }

g = reset_game()

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def spawn_obstacle(speed):
    """Return a new obstacle rect.  50% chance of each type.
    Type 0 – tall ground cactus (jump over)
    Type 1 – wide low log     (jump over differently, harder to time)
    Type 2 – floating bird    (duck under)
    """
    kind = random.randint(0, 2)
    if kind == 0:          # tall cactus
        h = random.randint(40, 70)
        return pygame.Rect(W, GROUND - h, 25, h), kind
    elif kind == 1:        # wide low log
        h = random.randint(20, 35)
        return pygame.Rect(W, GROUND - h, 55, h), kind
    else:                  # at head height
        h = 22
        y = GROUND - 50 - random.randint(30, 70)   # above player head
        return pygame.Rect(W, y, 40, h), kind

def draw_hearts(lives):
    for i in range(3):
        colour = HEART_C if i < lives else (100, 100, 100)
        x = W - 30 - i * 28
        pygame.draw.polygon(screen, colour, [
            (x,     22),
            (x - 9, 12), (x - 9,  6),
            (x,      9),
            (x + 9,  6), (x + 9, 12),
        ])

def draw_hud(score, high_score, lives):
    screen.blit(font_med.render(f"Score: {score}",      True, BLACK), (10, 10))
    screen.blit(font_sm .render(f"Best:  {high_score}", True, DARK),  (10, 42))
    draw_hearts(lives)

# ------------------------------------------------------------------
# Screens
# ------------------------------------------------------------------

def draw_menu():
    screen.fill(SKY)
    pygame.draw.rect(screen, GRASS, (0, GROUND, W, H - GROUND))

    title = font_big.render("ENDLESS RUNNER", True, DARK)
    screen.blit(title, (W // 2 - title.get_width() // 2, 60))

    sub = font_med.render("SPACE  –  Start", True, BLACK)
    screen.blit(sub, (W // 2 - sub.get_width() // 2, 150))

    quit_txt = font_med.render("Q  –  Quit", True, (100, 0, 0))
    screen.blit(quit_txt, (W // 2 - quit_txt.get_width() // 2, 195))

    if high_score > 0:
        hs = font_sm.render(f"Session best: {high_score}", True, DARK)
        screen.blit(hs, (W // 2 - hs.get_width() // 2, 248))


def draw_game_over(score, high_score):
    # semi-transparent overlay
    overlay = pygame.Surface((W, H), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 120))
    screen.blit(overlay, (0, 0))

    go  = font_big.render("GAME OVER", True, RED)
    screen.blit(go,  (W // 2 - go.get_width()  // 2, 70))

    sc  = font_med.render(f"Score: {score}",      True, WHITE)
    hs  = font_med.render(f"Best:  {high_score}", True, WHITE)
    screen.blit(sc,  (W // 2 - sc.get_width()  // 2, 148))
    screen.blit(hs,  (W // 2 - hs.get_width()  // 2, 182))

    restart = font_med.render("SPACE – Restart     Q – Quit", True, WHITE)
    screen.blit(restart, (W // 2 - restart.get_width() // 2, 230))


# ------------------------------------------------------------------
# Main loop
# ------------------------------------------------------------------

running = True

while running:
    clock.tick(60)

    # ---- Events ----
    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            running = False

        if e.type == pygame.KEYDOWN:
            if e.key == pygame.K_q:
                running = False

            if e.key in (pygame.K_SPACE, pygame.K_UP):
                if state == "menu":
                    g = reset_game()
                    state = "playing"

                elif state == "dead":
                    g = reset_game()
                    state = "playing"

                elif state == "playing" and g["on_ground"]:
                    g["vy"] = JUMP_VY
                    g["on_ground"] = False

    # ---- Update ----
    if state == "playing":
        # Physics
        g["vy"] += GRAVITY
        g["player"].y += int(g["vy"])
        if g["player"].y >= GROUND - 50:
            g["player"].y = GROUND - 50
            g["vy"]        = 0
            g["on_ground"] = True

        # Spawn obstacles
        g["spawn"] += 1
        if g["spawn"] > random.randint(60, 120):
            obs, kind = spawn_obstacle(g["speed"])
            g["obstacles"].append((obs, kind))
            g["spawn"] = 0

        # Move obstacles
        for obs, kind in g["obstacles"]:
            obs.x -= g["speed"]
        g["obstacles"] = [(o, k) for o, k in g["obstacles"] if o.right > 0]

        # Score & speed
        g["score"] += 1
        g["speed"]  = BASE_SPEED + g["score"] // 500

        # Invincibility countdown
        if g["invincible"] > 0:
            g["invincible"] -= 1

        # Collision
        if g["invincible"] == 0:
            for obs, kind in g["obstacles"]:
                if g["player"].colliderect(obs):
                    g["lives"]     -= 1
                    if g["lives"] > 0:
                        g["score"]  = g["score"] // 2
                    g["speed"]      = max(BASE_SPEED, g["speed"] - 1)
                    g["invincible"] = 90   # 1.5 seconds at 60 fps
                    # push player back to ground so they don't get stuck
                    g["player"].y  = GROUND - 50
                    g["vy"]        = 0
                    g["on_ground"] = True

                    if g["lives"] <= 0:
                        high_score = max(high_score, g["score"])
                        state = "dead"
                    break

    # ---- Draw ----
    screen.fill(SKY)
    pygame.draw.rect(screen, GRASS, (0, GROUND, W, H - GROUND))

    if state == "menu":
        draw_menu()

    elif state in ("playing", "dead"):
        # Player (flashes when invincible)
        if state == "playing":
            if g["invincible"] == 0 or (g["invincible"] // 6) % 2 == 0:
                pygame.draw.rect(screen, PLAYER_C, g["player"])
        else:
            pygame.draw.rect(screen, PLAYER_C, g["player"])

        # Obstacles
        for obs, kind in g["obstacles"]:
            colour = OBS2_C if kind == 2 else OBS_C
            pygame.draw.rect(screen, colour, obs)

        draw_hud(g["score"], high_score, g["lives"])

        if state == "dead":
            draw_game_over(g["score"], high_score)

    pygame.display.flip()

pygame.quit()
sys.exit()

SystemExit: 